# Análisis Exploratorio de Datos: Reseñas de Hoteles

En esta lección vamos a trabajar con un dataset real de **515,000 reseñas de hoteles en Europa**. El objetivo es aprender a:

1. Cargar y explorar un dataset grande
2. Entender las columnas y su utilidad
3. Calcular estadísticas relevantes
4. Preparar los datos para análisis de sentimiento

**Dataset**: [515K Hotel Reviews Data in Europe](https://www.kaggle.com/jiashenliu/515k-hotel-reviews-data-in-europe) (CC0: Public Domain)

## ¿Por qué explorar los datos primero?

Antes de aplicar NLP o ML, **siempre** debemos entender nuestros datos:

- ¿Qué columnas hay?
- ¿Hay valores faltantes?
- ¿Los valores tienen sentido?
- ¿Hay inconsistencias?

Saltarse este paso es la causa #1 de modelos que funcionan mal.

## Carga de datos

Primero necesitamos descargar el dataset de Kaggle y guardarlo en la carpeta `data/` del proyecto.

In [ ]:
import pandas as pd
import time

# Cargar el dataset
print("Cargando datos...")
start = time.time()

# Asegúrate de tener Hotel_Reviews.csv en la carpeta data/
df = pd.read_csv('../data/Hotel_Reviews.csv')

end = time.time()
print(f"Carga completada en {round(end - start, 2)} segundos")

## Exploración inicial

Veamos qué tenemos:

In [ ]:
# ¿Cuántas filas y columnas?
print(f"Forma del dataset: {df.shape}")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print()

# Primeras filas
df.head()

In [ ]:
# Información general del dataset
df.info()

## Entender las columnas

Las columnas se pueden agrupar en 4 categorías:

### Columnas del hotel
- `Hotel_Name`, `Hotel_Address`, `lat`, `lng`

### Columnas de meta-reseñas
- `Average_Score` — Puntuación promedio del hotel
- `Total_Number_of_Reviews` — Total de reseñas
- `Additional_Number_of_Scoring` — Reseñas con puntuación pero sin texto

### Columnas de reseñas
- `Negative_Review` / `Positive_Review` — Texto de la reseña
- `Reviewer_Score` — Puntuación del revisor (2.5 a 10)
- `Tags` — Descriptores del tipo de viaje, habitación, etc.

### Columnas del revisor
- `Reviewer_Nationality` — Nacionalidad del revisor
- `Total_Number_of_Reviews_Reviewer_Has_Given` — Cuántas reseñas ha escrito

In [ ]:
# Veamos las columnas disponibles
print("Columnas:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

---

## Ejercicio 1: Nacionalidades de los revisores

¿De dónde son las personas que dejan reseñas?

In [ ]:
# ¿Cuántas nacionalidades diferentes hay?
nationality_freq = df["Reviewer_Nationality"].value_counts()

print(f"Total de nacionalidades diferentes: {nationality_freq.size}")
print()
print("Top 10 nacionalidades con más reseñas:")
print(nationality_freq[:10].to_string())

### ¿Qué vemos?

- Reino Unido domina con ~245K reseñas (casi la mitad)
- EE.UU. es segundo con ~35K
- Esto tiene sentido: Booking.com es más popular en Europa

**Nota**: Los nombres de país tienen un espacio al principio — ¡cuidado con eso!

## Ejercicio 2: Hoteles más reseñados por nacionalidad

In [ ]:
# Para las top 10 nacionalidades, ¿cuál es el hotel más reseñado?
for nat in nationality_freq[:10].index:
    nat_df = df[df["Reviewer_Nationality"] == nat]
    freq = nat_df["Hotel_Name"].value_counts()
    print(f"{nat.strip():30s} → {freq.index[0]:45s} ({freq[0]:,} reseñas)")

## Ejercicio 3: Análisis de puntuaciones

El dataset tiene una columna `Average_Score` calculada por Booking.com. Pero nosotros podemos calcular la nuestra.

In [ ]:
# Calcular nuestro propio promedio de puntuación por hotel
df['Calc_Average_Score'] = round(df.groupby('Hotel_Name').Reviewer_Score.transform('mean'), 1)

# Comparar con el promedio original
df["Average_Score_Difference"] = df["Average_Score"] - df["Calc_Average_Score"]

# Ver los hoteles con mayor diferencia
review_scores_df = df.drop_duplicates(subset=["Hotel_Name"])
review_scores_df = review_scores_df.sort_values(by=["Average_Score_Difference"])

print("Hoteles con mayor diferencia entre promedio original y calculado:")
print()
display(review_scores_df[["Hotel_Name", "Average_Score", "Calc_Average_Score", "Average_Score_Difference"]].head(10))

### ¿Por qué difieren los promedios?

El `Average_Score` original parece basarse en un cálculo diferente (quizás incluye reseñas que no están en el dataset). Nosotros usaremos nuestro `Calc_Average_Score` porque es verificable.

## Ejercicio 4: Reseñas vacías

Algunos revisores dan puntuación pero no escriben texto. Veamos cuántos hay.

In [ ]:
# Contar reseñas vacías
no_negative = sum(df.Negative_Review == "No Negative")
no_positive = sum(df.Positive_Review == "No Positive")
both_empty = sum((df.Negative_Review == "No Negative") & (df.Positive_Review == "No Positive"))

print(f"Reseñas negativas vacías ('No Negative'): {no_negative:,}")
print(f"Reseñas positivas vacías ('No Positive'): {no_positive:,}")
print(f"Ambas vacías: {both_empty:,} ({both_empty/len(df)*100:.2f}% del total)")

### ¿Qué hacer con las reseñas vacías?

- **"No Negative"** en `Negative_Review` significa que el revisor no escribió nada negativo
- **"No Positive"** en `Positive_Review` significa que el revisor no escribió nada positivo
- Solo 127 personas (0.02%) no escribieron nada en ninguna de las dos

Estas "reseñas" vacías no son útiles para análisis de sentimiento, pero sí nos dicen algo: si alguien no escribe nada positivo ni negativo, probablemente su puntuación refleja su experiencia.

## Ejercicio 5: Distribución de puntuaciones

In [ ]:
# Distribución de puntuaciones de revisores
print("Distribución de puntuaciones:")
print(df["Reviewer_Score"].describe())
print()

# Histograma de puntuaciones
df["Reviewer_Score"].hist(bins=20, edgecolor='black')
import matplotlib.pyplot as plt
plt.xlabel("Puntuación")
plt.ylabel("Cantidad de reseñas")
plt.title("Distribución de puntuaciones de revisores")
plt.show()

## Resumen: Lo que aprendimos

| Hallazgo | Detalle |
|----------|--------|
| **Tamaño** | 515,738 reseñas, 17 columnas |
| **Nacionalidades** | 227 diferentes, Reino Unido domina |
| **Reseñas vacías** | ~127K sin texto negativo, ~36K sin positivo |
| **Promedios** | Los originales difieren de los calculados — usar los calculados |
| **Puntuaciones** | Van de 2.5 a 10, sesgadas hacia arriba |

### ¿Por qué importa esto?

Antes de hacer NLP, necesitamos:
1. **Limpiar** los datos (reseñas vacías, inconsistencias)
2. **Entender** qué columnas son útiles
3. **Verificar** que los valores calculados tienen sentido

En la próxima lección, filtraremos las columnas y aplicaremos análisis de sentimiento con NLTK VADER.

---

## Preguntas para reflexionar

1. ¿Por qué es problemático que Reino Unido tenga el 47% de las reseñas si el dataset es "de Europa"?

2. Si un hotel tiene un promedio de 8.5 pero solo 10 reseñas, ¿es confiable? ¿Qué tan importante es el número de reseñas?

3. ¿Por qué el dataset tiene un mínimo de 2.5 en lugar de 0? ¿Qué implicaciones tiene esto?